# Transaccional clientes procesamiento

In [0]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as f

In [0]:
spark = SparkSession.builder.appName("Transaccional clientes delta").getOrCreate()

## Data understanding

In [0]:
df = spark.read.format("parquet").load("/Volumes/workspace/default/transaccional_clientes_raw")
display(df.limit(10))

In [0]:
id_cols = ['cliente_id', 'agencia_id', 'ruta_id']

cat_cols = ['pais_cd', 'region_comercial_txt', 'tipo_cliente_cd', 'madurez_digital_cd', 'estrellas_txt', 'frecuencia_visitas_cd', 'canal_pedido_cd']

num_cols = ['facturacion_usd_val', 'materiales_distintos_val', 'cajas_fisicas']

date_cols = ['fecha_pedido_dt']

In [0]:
null_counts = df.select([f.count(f.when(f.col(c).isNull(), c)).alias(c) for c in df.columns])
display(null_counts)

No hay nulos puros, pero se deben revisar las columnas tipo string si hay equivalencias de nulos.

In [0]:
print('Cantidad de observaciones:', df.count())
print('Cantidad de clientes:', df.select('cliente_id').distinct().count())

Un cliente puede tener varios pedidos

In [0]:
for col in cat_cols + id_cols:
    distinct_count = df.select(col).distinct().count()
    print(f"\nValores distintos en {col}: {distinct_count}")
    counts = df.groupBy(col).count().orderBy(f.desc("count"))
    display(counts.limit(15))

La data parece estar bastante limpia.

In [0]:
for col in date_cols:
    min_date = df.agg(f.min(col)).first()[0]
    max_date = df.agg(f.max(col)).first()[0]
    print(f"\nColumna: {col}")
    print(f"Fecha mínima: {min_date}")
    print(f"Fecha máxima: {max_date}")

    # Primeras 10 fechas y sus conteos
    first_10 = df.groupBy(col).count().orderBy(f.asc(col)).limit(10)
    print("Primeras 10 fechas y sus conteos:")
    display(first_10)

    # Últimas 10 fechas y sus conteos
    last_10 = df.groupBy(col).count().orderBy(f.desc(col)).limit(10)
    print("Últimas 10 fechas y sus conteos:")
    display(last_10)

In [0]:
for col in date_cols:
    # Extraer año y mes
    df_month = df.withColumn("year_month", f.date_trunc("month", f.col(col)))
    
    # Conteos por mes
    months_counts = (
        df_month.groupBy("year_month")
        .count()
        .orderBy(f.asc("year_month"))
    )
    
    display(months_counts)

Las fechas parecen estar en orden. Agosto 2024 no esta completo.

## Limpieza y Calidad de Datos

¿Qué variables están a nivel cliente?

In [0]:
# Verificar si un cliente tiene más de un valor distinto en varias columnas categóricas
cols_to_check = [
    ("madurez_digital_cd", "madurez_distinta_count"),
    ("tipo_cliente_cd", "tipo_distinto_count"),
    ("pais_cd", "pais_distinto_count"),
    ("region_comercial_txt", "region_distinta_count"),
    ("frecuencia_visitas_cd", "frecuencia_distinta_count"),
    ("estrellas_txt", "estrellas_distinta_count"),
    ("canal_pedido_cd", "canal_distinto_count"),
    ("ruta_id", "ruta_distinta_count"),
    ("agencia_id", "agencia_distinta_count"),
    ("facturacion_usd_val", "facturacion_distinta_count"),
    ("materiales_distintos_val", "materiales_distinta_count"),
    ("cajas_fisicas", "cajas_distinta_count")
]

for col, alias in cols_to_check:
    clientes_varios = (
        df.groupBy("cliente_id")
        .agg(f.countDistinct(col).alias(alias))
        .filter(f.col(alias) > 1)
    )
    print(f"Cantidad de clientes que cambiaron de {col}: {clientes_varios.count()}")

La mayoría de las variables están a nivel cliente. Solo cambian las numéricas y el canal.

## Delta table

In [0]:
df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.transaccional_clientes")

In [0]:
df_delta = spark.read.table("workspace.default.transaccional_clientes")
display(df_delta.limit(10))